In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

In [3]:
def load_and_explore_data(file_path):
    """Loads dataset and prints basic exploration metrics."""
    try:
        df = pd.read_csv(file_path)
        print(f"Dataset loaded successfully. Shape: {df.shape}")
        print("--- First 5 Rows ---")
        print(df.head())
        print("--- Data Info ---")
        print(df.info())
        print("--- Missing Values ---")
        print(df.isnull().sum())
        return df
    except Exception as e:
        print(f"Failed to load data: {e}")
        return None

In [4]:
def clean_text(text):
    """Cleans raw text by removing noise, punctuation, and stopwords."""
    if not isinstance(text, str):
        return ""

    # Lowercase text
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)
    # Remove stopwords
    words = text.split()
    cleaned_words = [word for word in words if word not in STOPWORDS]

    return ' '.join(cleaned_words)

In [5]:
def preprocess_dataset(df, text_column):
    """Applies cleaning to the entire dataframe."""
    print(f"Cleaning text in column: '{text_column}'...")

    # Drop rows where the text column is completely missing
    df = df.dropna(subset=[text_column]).copy()

    # Apply the cleaning function
    df['cleaned_text'] = df[text_column].apply(clean_text)
    return df

In [6]:
def vectorize_text(texts, max_words=10000, max_len=200):
    """Tokenizes and pads sequences for deep learning models."""
    print("Tokenizing and vectorizing text...")

    # Initialize Tokenizer
    tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
    tokenizer.fit_on_texts(texts)

    # Convert text to integer sequences
    sequences = tokenizer.texts_to_sequences(texts)

    # Pad sequences to ensure uniform length for the neural network
    padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

    return padded_sequences, tokenizer

In [7]:
TRAIN_PATH = "/content/drive/MyDrive/prodatatext/train.csv"

In [8]:
test = "/content/drive/MyDrive/prodatatext/test.csv"

In [9]:
df = pd.read_csv(test)

df.head(10)

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.
5,0001ea8717f6de06,Thank you for understanding. I think very high...
6,00024115d4cbde0f,Please do not add nonsense to Wikipedia. Such ...
7,000247e83dcc1211,:Dear god this site is horrible.
8,00025358d4737918,""" \n Only a fool can believe in such numbers. ..."
9,00026d1092fe71cc,== Double Redirects == \n\n When fixing double...


In [ ]:
df_train = load_and_explore_data(TRAIN_PATH)

Dataset loaded successfully. Shape: (159571, 8)
--- First 5 Rows ---
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0        0       0       0              0  
4             0        0       0       0              0  
--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570


In [ ]:
TEXT_COLUMN = "comment_text"

In [ ]:
df_train_cleaned = preprocess_dataset(df_train, TEXT_COLUMN)

Cleaning text in column: 'comment_text'...


In [ ]:
# Assuming df_train is the CORRECT file with the 6 label columns
label_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Extract the labels into a numpy array (This is your y)
y_train_array = df_train[label_columns].values

# Extract and vectorize the text (This is your X)
X_train_vectorized, fitted_tokenizer = vectorize_text(df_train['comment_text'])

print(f"X shape: {X_train_vectorized.shape}") # Should be (159571, 200)
print(f"y shape: {y_train_array.shape}")      # Should be (159571, 6)

Tokenizing and vectorizing text...
X shape: (159571, 200)
y shape: (159571, 6)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import numpy as np

In [ ]:
def build_toxicity_model(vocab_size, max_len, embedding_dim=128, num_classes=6):
    """
    Builds a Bidirectional LSTM model for multi-label text classification.
    """
    print("Constructing Bidirectional LSTM Architecture...")

    model = Sequential([
        Input(shape=(max_len,)),

        Embedding(input_dim=vocab_size, output_dim=embedding_dim),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.2),
        Bidirectional(LSTM(64)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='sigmoid')
    ])
    # CRITICAL BLIND SPOT 2: Loss MUST be binary_crossentropy, not categorical.
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(name='auc', multi_label=True),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            'accuracy' # Included for baseline, but do not trust accuracy on imbalanced data.
        ]
    )

    model.summary()
    return model

In [ ]:
def train_and_save_model(model, X_train, y_train, X_val, y_val, batch_size=32, epochs=5):
    """
    Trains the model with callbacks to prevent wasted computing time.
    """
    print("Initiating training sequence...")

    # Stop training if the validation loss stops improving
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    # Save the model automatically so you don't lose it if your kernel crashes
    model_save_path = 'toxicity_model.h5'
    checkpoint = ModelCheckpoint(model_save_path, monitor='val_loss', save_best_only=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=batch_size,
        epochs=epochs,
        callbacks=[early_stop, checkpoint]
    )

    print(f"Training complete. Best model saved to: {model_save_path}")
    return history

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout, Input
from sklearn.model_selection import train_test_split

In [ ]:
# --- EXECUTION BLOCK ---
if __name__ == "__main__":
    # You MUST have X_train_vectorized and y_train_array defined from train.csv.

    print("Splitting data into training and validation sets...")

    # 80% for training, 20% for validation
    X_train_split, X_val_vectorized, y_train_split, y_val_array = train_test_split(
        X_train_vectorized,
        y_train_array,
        test_size=0.2,
        random_state=42
    )

    print(f"Training data shape: {X_train_split.shape}")
    print(f"Validation data shape: {X_val_vectorized.shape}")

    # Build the model
    # (Assuming vocab_size=10000, max_len=200, num_classes=6)
    model = build_toxicity_model(vocab_size=10000, max_len=200, num_classes=6)

    # NOW you can train the model
    history = train_and_save_model(model, X_train_split, y_train_split, X_val_vectorized, y_val_array)

Splitting data into training and validation sets...
Training data shape: (127656, 200)
Validation data shape: (31915, 200)
Constructing Bidirectional LSTM Architecture...


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_6 (Bidirectional) │ (None, 200, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 200, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_7 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,486,278 (5.67 MB)

 Trainable params: 1,486,278 (5.67 MB)

 Non-trainable params: 0 (0.00 B)

Initiating training sequence...
Epoch 1/5
3989/3990 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9264 - auc: 0.8650 - loss: 0.0939 - precision: 0.7012 - recall: 0.4088

3990/3990 ━━━━━━━━━━━━━━━━━━━━ 158s 38ms/step - accuracy: 0.9752 - auc: 0.9324 - loss: 0.0661 - precision: 0.7874 - recall: 0.5535 - val_accuracy: 0.9941 - val_auc: 0.9663 - val_loss: 0.0497 - val_precision: 0.8057 - val_recall: 0.6606
Epoch 2/5
3989/3990 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9919 - auc: 0.9659 - loss: 0.0477 - precision: 0.8219 - recall: 0.6657

3990/3990 ━━━━━━━━━━━━━━━━━━━━ 152s 38ms/step - accuracy: 0.9924 - auc: 0.9662 - loss: 0.0478 - precision: 0.8193 - recall: 0.6669 - val_accuracy: 0.9941 - val_auc: 0.9658 - val_loss: 0.0482 - val_precision: 0.7975 - val_recall: 0.6935
Epoch 3/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 151s 38ms/step - accuracy: 0.9894 - auc: 0.9721 - loss: 0.0435 - precision: 0.8304 - recall: 0.6953 - val_accuracy: 0.9940 - val_auc: 0.9649 - val_loss: 0.0495 - val_precision: 0.8221 - val_recall: 0.6689
Epoch 4/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 147s 37ms/step - accuracy: 0.9822 - auc: 0.9770 - loss: 0.0398 - precision: 0.8398 - recall: 0.7177 - val_accuracy: 0.9940 - val_auc: 0.9606 - val_loss: 0.0496 - val_precision: 0.8092 - val_recall: 0.6763
Training complete. Best model saved to: toxicity_model.h5


In [ ]:
import pickle

with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(fitted_tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Tokenizer saved successfully.")

Tokenizer saved successfully.
